## Problem 7

In `./data/ames_prices.csv`, create an indicator that equals 1 when `Central.Air == 'Y'` and 0 otherwise.

**a.** Estimate the proportion of homes with central air.

**b.** Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.

**c.** Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

**d.** Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.

**e.** Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.

**Answer:**



## 7a

In [1]:
import pandas as pd
import numpy as np

ames = pd.read_csv("./data/ames_prices.csv")
ames.head()


,Order,PID,area,price,MS.SubClass,MS.Zoning,Lot.Frontage,Lot.Area,Street,Alley,...,Screen.Porch,Pool.Area,Pool.QC,Fence,Misc.Feature,Misc.Val,Mo.Sold,Yr.Sold,Sale.Type,Sale.Condition
0,1,526301100,1656,215000,20,RL,141.0,31770,Pave,NaN,...,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal
1,2,526350040,896,105000,20,RH,80.0,11622,Pave,NaN,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
2,3,526351010,1329,172000,20,RL,81.0,14267,Pave,NaN,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
3,4,526353030,2110,244000,20,RL,93.0,11160,Pave,NaN,...,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal
4,5,527105010,1629,189900,60,RL,74.0,13830,Pave,NaN,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [2]:
ames["central_air"] = (ames["Central.Air"] == "Y").astype(int)
p_hat = ames["central_air"].mean()

print("Proportion with central air:", p_hat)


Proportion with central air: 0.9331058020477816


Approximately 93.3% of homes in the dataset have central ar conditioning.

## 7b

In [3]:

rng = np.random.default_rng(6021)
B = 2000

central_air = ames["central_air"].to_numpy()
n = len(central_air)
boot_props = []

for i in range(B):
    sample = rng.choice(central_air, size=n, replace=True)
    boot_props.append(sample.mean())

boot_props = np.array(boot_props)
print("Number of bootstrap estimates:", len(boot_props))


Number of bootstrap estimates: 2000


## 7c

In [4]:
boot_se = np.std(boot_props, ddof=1)
print("Bootstrap standard error:", boot_se)


Bootstrap standard error: 0.0046837039050191065


## 7d

In [5]:
critical_values = {
    "90%": 1.645,
    "95%": 1.96,
    "99%": 2.576
}

for level, z in critical_values.items():
    lower = p_hat - z * boot_se
    upper = p_hat + z * boot_se
    print(f"{level} CI: ({lower:.4f}, {upper:.4f})")


90% CI: (0.9254, 0.9408)
95% CI: (0.9239, 0.9423)
99% CI: (0.9210, 0.9452)


## 7e

We reject the null hypothesis at the 10%, 5%, and 1% significance levels because 0.90 is outside all three confidence intervals.

## Problem 8

In `./data/ames_prices.csv`, estimate the Pearson correlation between `np.arcsinh(area)` and `np.arcsinh(price)`.

**a.** Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.

**b.** Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

**c.** Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

**d.** Use each confidence interval to evaluate $H_0:\rho=0$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret the association without making a causal claim.

**e.** Briefly explain why `area` and `price` must remain paired during resampling.

**Answer:**



## 8a

In [21]:

ames["area_transformed"] = np.arcsinh(ames["area"])
ames["price_transformed"] = np.arcsinh(ames["price"])

corr = ames["area_transformed"].corr(ames["price_transformed"])
print("Original correlation:", corr)


Original correlation: 0.7231661530091356


In [ ]:

rng = np.random.default_rng(6021)
B = 2000

paired_data = ames[["area_transformed", "price_transformed"]].dropna().to_numpy()
n = len(paired_data)
boot_corrs = []

for i in range(B):
    indices = rng.choice(n, size=n, replace=True)
    sample = paired_data[indices]
    boot_corrs.append(np.corrcoef(sample[:, 0], sample[:, 1])[0, 1])

boot_corrs = np.array(boot_corrs)
print("Seed:", 6021)
print("Bootstrap replications:", len(boot_corrs))


Seed: 6021
Bootstrap replications: 2000


## 8b

In [9]:
boot_se_corr = np.std(boot_corrs, ddof=1)
print("Bootstrap standard error:", boot_se_corr)

Bootstrap standard error: 0.010031740907783575


## 8c

In [10]:
critical_values = {
    "90%": 1.645,
    "95%": 1.96,
    "99%": 2.576
}

for level, z in critical_values.items():
    lower = corr - z * boot_se_corr
    upper = corr + z * boot_se_corr
    print(f"{level} CI: ({lower:.4f}, {upper:.4f})")


90% CI: (0.7067, 0.7397)
95% CI: (0.7035, 0.7428)
99% CI: (0.6973, 0.7490)


## 8d

We reject the null hypothesis at the 10%, 5%, and 1% signiifcance levels because zero is outside of the 3 confidence intervals.

## 8e

Area and price must remain paired becuase they are both variables that pertain to the same house. Becasue fo this theres a relationship between the two variables and separating them would affct the correlation estimate.

## Problem 9

In `./data/metabric.csv`, use `Overall Survival (Months)` to estimate the mean recorded time to death or last follow-up. Include both living and deceased patients.

**a.** Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

**b.** Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).

**c.** Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

**d.** Use each confidence interval to evaluate $H_0:\mu=120$ months. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.

**e.** Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.

**Answer:**



## 9a

In [11]:
import pandas as pd
import numpy as np

metabric = pd.read_csv("./data/metabric.csv")
metabric.head()

,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0


In [12]:
survival = metabric["Overall Survival (Months)"].dropna().to_numpy()
mean_survival = survival.mean()
print("Mean survival time:", mean_survival)


Mean survival time: 129.09429138744105


In [13]:
rng = np.random.default_rng(6021)
B = 2000

n = len(survival)
boot_means = []

for i in range(B):
    sample = rng.choice(survival, size=n, replace=True)
    boot_means.append(sample.mean())

boot_means = np.array(boot_means)

print("Seed:", 6021)
print("Bootstrap replications:", len(boot_means))


Seed: 6021
Bootstrap replications: 2000


## 9b

In [14]:
boot_se_survival = np.std(boot_means, ddof=1)
print("Bootstrap standard error:", boot_se_survival)

Bootstrap standard error: 2.2046634528744575


## 9c

In [15]:
critical_values = {
    "90%": 1.645,
    "95%": 1.96,
    "99%": 2.576
}

for level, z in critical_values.items():
    lower = mean_survival - z * boot_se_survival
    upper = mean_survival + z * boot_se_survival
    print(f"{level} CI: ({lower:.2f}, {upper:.2f})")


90% CI: (125.47, 132.72)
95% CI: (124.77, 133.42)
99% CI: (123.42, 134.77)


## 9d

We reject the null hypothesis at the 10%, 5%, and 1% significance levels because 120 is not in any of the confidence intervals.

## 9e

The mean survival time is approximately 129.09 months or 10.76 years. The 90% CI is 125.47 to 132.72 months, or 10.46 to 11.06 years. The 95% CI is 124.77 to 133.42 months, or 10.40 to 11.12 years. The 99% CI is 123.42 to 134.77 months, or 10.28 to 11.23 years. These results do not estimate the unrestricted mean lifetime because some patients are still alive. Fore these patients, their recorded survival times only go up to their last follow up.

## Problem 10

In `./data/nhanes_data_17_18.csv`, select respondents who are at least 18 years old and have an observed value for `SleepHoursWeekdaysOrWorkdays`. Estimate their mean weekday sleep.

**a.** Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.

**b.** Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (`ddof=1`).

**c.** Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.

**d.** Use each confidence interval to evaluate $H_0:\mu=8$ hours. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.

**e.** Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

**Answer:**



## 10a

In [16]:
import pandas as pd
import numpy as np

nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")
nhanes.head()

/var/folders/ll/0vvpn9_55vl1d38m8qn8m_vm0000gn/T/ipykernel_83543/3962993804.py:4: DtypeWarning: Columns (142) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("./data/nhanes_data_17_18.csv")


,SEQN,GeneralHealthCondition,EverBreastfedOrFedBreastmilk,AgeStoppedBreastfeedingdays,AgeFirstFedFormuladays,AgeStoppedReceivingFormuladays,AgeStartedOtherFoodbeverage,AgeFirstFedMilkdays,TypeOfMilkFirstFedWholeMilk,TypeOfMilkFirstFed2Milk,...,DaysSmokedCigsDuringPast30Days,AvgCigarettesdayDuringPast30Days,TriedToQuitSmoking,TimesStoppedSmokingCigarettes,HowLongWereYouAbleToStopSmoking,UnitOfMeasureDayweekmonthyear_2_SMQ,CurrentSelfreportedHeightInches,CurrentSelfreportedWeightPounds,TriedToLoseWeightInPastYear,TimesLost10LbsOrMoreToLoseWeight
0,93703.0,NaN,1.0,273.0,1.0,365.0,152.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,1.0,60.0,3.0,365.0,126.0,365.0,Whole or regular milk,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,63.0,165.0,0.0,11 times or more
3,93706.0,Very good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,145.0,0.0,Never
4,93707.0,Good,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
adults = nhanes[
    (nhanes["AgeInYearsAtScreening"] >= 18) &
    (nhanes["SleepHoursWeekdaysOrWorkdays"].notna())
]
sleep = adults["SleepHoursWeekdaysOrWorkdays"].to_numpy()
mean_sleep = sleep.mean()

print("Mean weekday sleep:", mean_sleep)

Mean weekday sleep: 7.62124248496994


In [18]:
rng = np.random.default_rng(6021)
B = 2000
boot_sleep = []

for i in range(B):
    sample = rng.choice(sleep, size=len(sleep), replace=True)
    boot_sleep.append(sample.mean())

boot_sleep = np.array(boot_sleep)

print("Seed:", 6021)
print("Bootstrap replications:", len(boot_sleep))

Seed: 6021
Bootstrap replications: 2000


## 10b

In [19]:
boot_se_sleep = np.std(boot_sleep, ddof=1)
print("Bootstrap standard error:", boot_se_sleep)

Bootstrap standard error: 0.02210212051152941


## 10c

In [20]:
critical_values = {
    "90%": 1.645,
    "95%": 1.96,
    "99%": 2.576
}
for level, z in critical_values.items():
    lower = mean_sleep - z * boot_se_sleep
    upper = mean_sleep + z * boot_se_sleep
    print(f"{level} CI: ({lower:.4f}, {upper:.4f})")

90% CI: (7.5849, 7.6576)
95% CI: (7.5779, 7.6646)
99% CI: (7.5643, 7.6782)


## 10d

We reject the null hypothesis at the 10%, 5%, and 1% siginificance levels because 8 hours is outside all of the confindence intervals.

## 10e

The average weekday sleep is about 7.62 hours and is also less than 8 hours. This is statically significant because the 3 confidence intervals do not include 8 hours. Context is required to see if these resutls are practically significant. The results are not nationally representative because the survey design was ignored.

I used gen ai to turn the orginal markdown into an ipynb file and to troubleshoot codig errors. 